# CNU 학내정보 RAG 챗봇 Ver1 - 코랩 마스터 노트북

충남대학교 학생/교직원 12개 카테고리 질문에 환각 없이 답변하는 RAG 챗봇.

## 실행 방법
1. 런타임 → 런타임 유형 변경 → T4 GPU 선택
2. 셀 3에서 GitHub URL을 본인 레포로 교체
3. 런타임 → 모두 실행 (예상 소요 시간: 15~30분)

## 필수 환경변수
- `ANTHROPIC_API_KEY`: Claude API 키 (질문 생성, LLM judge 필요)
- `QUESTION_LIMIT`: 질문 풀 생성 수 (기본 500, 풀스케일 10000)
- `JUDGE_MOCK`: 1이면 API 없이 mock 점수 (테스트용)

## 1. 환경 셋업

In [13]:

!git clone https://github.com/Longarden/cnu-llm-bot.git
%cd cnu-llm-bot

Cloning into 'cnu-llm-bot'...
remote: Enumerating objects: 79, done.
remote: Counting objects: 100% (79/79), done.
remote: Compressing objects: 100% (76/76), done.
remote: Total 79 (delta 1), reused 79 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (79/79), 57.53 KiB | 6.39 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/cnu-llm-bot


In [14]:
import os

os.environ["ANTHROPIC_API_KEY"] = "여기에_네_API키"
os.environ["JUDGE_MOCK"] = "1"
os.environ["DISTILL"] = "0"
os.environ["QUESTION_LIMIT"] = "500"
os.environ["MODEL_FALLBACK"] = "3b"


In [15]:
!pip install -q -r requirements.txt

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.3/74.3 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 112.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 501.9/501.9 kB 44.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 104.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.7/831.7 kB 56.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 29.8 MB/s eta 0:00:0

## 2. 데이터 수집 (크롤링)

12개 카테고리 크롤러를 병렬 실행합니다. 사이트 오류 시 `_fallback()` 더미 데이터로 자동 폴백됩니다.

In [16]:
from crawlers import run_all_crawlers
docs = run_all_crawlers()
print(f"수집된 문서: {len(docs)}")

[dining] 크롤 실패: 404 Client Error: Not Found for url: https://www.cnu.ac.kr/life/restMenu.do
[A_dining] 5건 수집
[A_library] 10건 수집
[A_shuttle] 5건 수집
[B_academic] 7건 수집
[C_administration] 6건 수집
[D_scholarship] 6건 수집
[E_career] 6건 수집
[F_department] 10건 수집
[G_dormitory] 5건 수집
[G_student_life] 5건 수집
[H_facilities] 7건 수집
[I_international] 5건 수집
[J_extracurricular] 5건 수집
[notices] https://www.cnu.ac.kr/bbs/CNU_40/list.do 실패: 404 Client Error: Not Found for url: https://www.cnu.ac.kr/bbs/CNU_40/list.do
[notices] https://www.cnu.ac.kr/bbs/CNU_60/list.do 실패: 404 Client Error: Not Found for url: https://www.cnu.ac.kr/bbs/CNU_60/list.do
[notices] https://computer.cnu.ac.kr/computer/notice/bachelor.do 실패: HTTPSConnectionPool(host='computer.cnu.ac.kr', port=443): Read timed out. (read timeout=10)
[notices] https://computer.cnu.ac.kr/computer/notice/notice.do 실패: HTTPSConnectionPool(host='computer.cnu.ac.kr', port=443): Read timed out. (read timeout=10)
[notices] https://computer.cnu.ac.kr/computer/not

## 3. LLM Distillation + Dedup (선택)

`DISTILL=1`로 설정하면 Claude Haiku로 핵심 내용을 추출합니다 (API 비용 발생).
기본값 `0`은 원문 그대로 사용합니다.

In [17]:
import os
os.environ["DISTILL"] = "0"  # 비용 절약. 1로 바꾸면 Claude Haiku distillation 활성

from crawler_pipeline.distiller import distill_all
from crawler_pipeline.deduplicator import dedup

distilled = distill_all(docs) if os.environ.get("DISTILL") == "1" else docs
deduped = dedup(distilled)
print(f"중복 제거 후: {len(deduped)}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

[deduplicator] 96 → 96건 (제거 0)
중복 제거 후: 96


## 4. 벡터 DB 구축

BGE-M3 임베딩 + Chroma persistent DB에 청크를 저장합니다.
모든 청크에 메타데이터 6종(source_url, data_category, last_crawled_at, valid_until, freshness_tier, original_text)이 강제 저장됩니다.

In [18]:
from embedding.vector_store import build_vector_db, count
build_vector_db(deduped)
print(f"벡터 DB 청크 수: {count()}")

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

[vector_store] 96/96 저장
[vector_store] 완료. 총 96건
벡터 DB 청크 수: 96


## 5. 1만 질문 풀 생성 (옵션) + 평가셋 200

`QUESTION_LIMIT=500`은 빠른 테스트용. `10000`으로 바꾸면 풀스케일 생성 (약 20~30분).
평가셋 200개는 스펙 분포로 추출: 시간성 50 / 학사 40 / 행정+장학 30 / 취업 25 / 학과 25 / 생활+시설 20 / 도메인 밖 10.

In [19]:
import os
os.environ["QUESTION_LIMIT"] = "500"  # 풀스케일은 "10000"

from question_generation.question_pool_builder import generate_question_pool
from eval.testset_builder import build_testset

generate_question_pool(output_path="data/questions/question_pool.jsonl")
build_testset(
    question_pool_path="data/questions/question_pool.jsonl",
    output_path="data/testset.jsonl",
    n=200
)

  API 오류 (학식 메뉴 및 운영시간): 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → 더미 사용
[A] 학식 메뉴 및 운영시간: 10개 생성 (누적 10)
  API 오류 (도서관 운영시간 및 좌석): 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → 더미 사용
[A] 도서관 운영시간 및 좌석: 10개 생성 (누적 20)
  API 오류 (셔틀버스 시간표): 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → 더미 사용
[A] 셔틀버스 시간표: 10개 생성 (누적 30)
  API 오류 (오늘/내일 일정 조회): 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → 더미 사용
[A] 오늘/내일 일정 조회: 10개 생성 (누적 40)
  API 오류 (주간 운영 변경 공지): 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → 더미 사용
[A] 주간 운영 변경 공지: 10개 생성 (누적 50)
  API 오류 (수강신청 방법 및 일정): 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → 더미 사용
[B] 수강신청 방법 및 일정: 10개 생성 (누적 60)
  API 오류 (졸업 요건 및 절차): 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → 더미 사용
[B] 졸업 요건 및 절차: 

[{'id': 'eval_0001',
  'category_id': 'A',
  'category_name': '시간성',
  'subcategory': '학식 메뉴 및 운영시간',
  'question': '충남대 학식 메뉴 및 운영시간에 대해 알려주세요.',
  'expected_answer': None},
 {'id': 'eval_0002',
  'category_id': 'A',
  'category_name': '시간성',
  'subcategory': '오늘/내일 일정 조회',
  'question': '오늘/내일 일정 조회 관련해서 문의드립니다.',
  'expected_answer': None},
 {'id': 'eval_0003',
  'category_id': 'A',
  'category_name': '시간성',
  'subcategory': '주간 운영 변경 공지',
  'question': '주간 운영 변경 공지 관련해서 문의드립니다.',
  'expected_answer': None},
 {'id': 'eval_0004',
  'category_id': 'A',
  'category_name': '시간성',
  'subcategory': '셔틀버스 시간표',
  'question': '충남대에서 셔틀버스 시간표 어디서 할 수 있나요?',
  'expected_answer': None},
 {'id': 'eval_0005',
  'category_id': 'A',
  'category_name': '시간성',
  'subcategory': '도서관 운영시간 및 좌석',
  'question': '충남대 도서관 운영시간 및 좌석 담당 부서가 어디인가요?',
  'expected_answer': None},
 {'id': 'eval_0006',
  'category_id': 'A',
  'category_name': '시간성',
  'subcategory': '도서관 운영시간 및 좌석',
  'question': '도서관 운영시간 및 좌석 

## 6. 모델 로드 + GPU 메모리 측정

Qwen2.5-7B-Instruct-AWQ (4bit)를 로드합니다. OOM 시 자동으로 3B 모델로 폴백됩니다.
목표: 추론 메모리 < 12GB (T4 15GB 한도에 3GB 마진).

In [20]:
from generation.llm import load_llm, print_gpu_memory
llm = load_llm()
print_gpu_memory()  # < 12GB 확인

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'top_p', 'temperature', 'do_sample', 'repetition_penalty'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


GPU 메모리: 7.87GB 사용 / 7.94GB 예약 / 14.56GB 전체


## 7. 교수님 인터페이스 데모 (질문 파일 in → 답변 파일 out)

`answer_questions(questions_path, answers_path)` 함수가 교수님 제출 인터페이스입니다.
샘플 5개 질문으로 end-to-end 동작을 확인합니다.

In [21]:
from interface.answer_questions import answer_questions
answer_questions("tests/sample_questions.jsonl", "data/sample_answers.jsonl")
!cat data/sample_answers.jsonl

[answer_questions] 1/5: 1학생회관 오늘 점심 메뉴 알려줘


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[hybrid_retriever] dense 검색 실패: Expected operand value to be an int or a float for operator $gte, got 2026-05-19 in query.
[answer_questions] 2/5: 졸업하려면 몇 학점 필요해?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[answer_questions] 3/5: 본 적 없는 정보 검색 테스트


...

[answer_questions] 4/5: 오늘 노벨상 누구야?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[hybrid_retriever] dense 검색 실패: Expected operand value to be an int or a float for operator $gte, got 2026-05-19 in query.
[answer_questions] 5/5: 어제 도서관 자리 있었어?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[answer_questions] 5개 답변 저장 -> data/sample_answers.jsonl
{"id": "q1", "question": "1학생회관 오늘 점심 메뉴 알려줘", "answer": "제공된 자료에 해당 정보가 없습니다.\n관련 학과 사무실이나 충남대학교 포털(portal.cnu.ac.kr)을 이용해 주세요.", "sources": [], "rejected": true, "latency_ms": 236}
{"id": "q2", "question": "졸업하려면 몇 학점 필요해?", "answer": "제공된 자료에 해당 정보가 없습니다. (검색 유사도: 0.02)\n더 구체적인 질문으로 다시 시도하거나 학교 포털을 확인해 주세요.", "sources": ["https://plus.cnu.ac.kr", "https://www.cnu.ac.kr/extracurricular", "https://www.cnu.ac.kr/administration"], "rejected": true, "latency_ms": 91}
{"id": "q3", "question": "본 적 없는 정보 검색 테스트", "answer": "제공된 자료에 해당 정보가 없습니다. (검색 유사도: 0.02)\n더 구체적인 질문으로 다시 시도하거나 학교 포털을 확인해 주세요.", "sources": ["https://library.cnu.ac.kr"], "rejected": true, "latency_ms": 62}
{"id": "q4", "question": "오늘 노벨상 누구야?", "answer": "제공된 자료에 해당 정보가 없습니다.\n관련 학과 사무실이나 충남대학교 포털(portal.cnu.ac.kr)을 이용해 주세요.", "sources": [], "rejected": true, "latency_ms": 61}
{"id": "q5", "question": "어제 도서관 자리 있었어?", "answer": "제공된 자료에 해당 정보가 없습니다. (검색 유사도: 0.02

## 8. 평가셋 200개 실행 + LLM-as-judge

3축 평가 (accuracy / relevance / domain_compliance, 각 0~5점).
`JUDGE_MOCK=1`로 설정하면 API 없이 더미 점수로 테스트 가능합니다.

In [22]:
import os
os.environ["JUDGE_MOCK"] = "0"  # 1 로 바꾸면 mock 모드 (API 비용 없음)

from interface.answer_questions import answer_questions
from eval.evaluator import run_evaluation

answer_questions("data/testset.jsonl", "data/testset_answers.jsonl")
results = run_evaluation(
    testset_path="data/testset.jsonl",
    answers_path="data/testset_answers.jsonl"
)
print(results)

[answer_questions] 1/200: 충남대 학식 메뉴 및 운영시간에 대해 알려주세요.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[answer_questions] 2/200: 오늘/내일 일정 조회 관련해서 문의드립니다.


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[hybrid_retriever] dense 검색 실패: Expected operand value to be an int or a float for operator $gte, got 2026-05-19 in query.
[answer_questions] 3/200: 주간 운영 변경 공지 관련해서 문의드립니다.


...

[answer_questions] 199/200: 김치찌개 레시피 알려줘


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[answer_questions] 200/200: 삼성전자 주가가 얼마예요?


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[answer_questions] 200개 답변 저장 -> data/testset_answers.jsonl
평가 시작: 200개 질문
Judge API 오류: 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → mock 반환
Judge API 오류: 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → mock 반환
Judge API 오류: 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → mock 반환
Judge API 오류: 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → mock 반환
Judge API 오류: 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → mock 반환
Judge API 오류: 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → mock 반환
Judge API 오류: 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → mock 반환
Judge API 오류: 'ascii' codec can't encode characters in position 0-2: ordinal not in range(128) → mock 반환
Judge API 오류: 'ascii' codec can't encode characters in position 0-2: ordinal not in r

## 9. 추론 속도 벤치마크 (동점자 변별 대비)

50개 질문 기준 평균 응답 시간(avg_latency_ms)과 GPU 메모리를 측정합니다.

In [23]:
from interface.benchmark import benchmark_inference
bench = benchmark_inference("data/testset.jsonl", n=50)
print(bench)

GPU 메모리: 7.87GB 사용 / 7.95GB 예약 / 14.56GB 전체


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  충남대 학식 메뉴 및 운영시간에 대해 알려주세요.: 53ms


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[hybrid_retriever] dense 검색 실패: Expected operand value to be an int or a float for operator $gte, got 2026-05-19 in query.
  오늘/내일 일정 조회 관련해서 문의드립니다.: 47ms


...

  주간 운영 변경 공지 방법이 어떻게 되나요?: 66ms


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

[hybrid_retriever] dense 검색 실패: Expected operand value to be an int or a float for operator $gte, got 2026-05-19 in query.
  충남대 오늘/내일 일정 조회에 대해 알려주세요.: 59ms


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

  셔틀버스 시간표 관련해서 문의드립니다.: 74ms

[benchmark] avg=65ms | p50=64ms | p95=79ms | gpu_peak=8062MB
GPU 메모리: 7.87GB 사용 / 7.95GB 예약 / 14.56GB 전체
{'n': 50, 'avg_latency_ms': 65.0, 'p50_latency_ms': 64.4, 'p95_latency_ms': 78.9, 'gpu_memory_peak_mb': 8061.8}


## 10. Gradio 데모 (시연용)

`share=True`로 실행되므로 `*.gradio.live` 공개 URL이 발급됩니다.
상위 5팀 발표 시연 및 데모에 사용합니다.

In [24]:
from app.gradio_app import launch_app
launch_app()  # share=True → *.gradio.live URL 발급

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e69c48eebe20a81563.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Gradio Blocks instance: 35 backend functions
--------------------------------------------
fn_index=0
 inputs:
 |-<gradio.components.dataset.Dataset object at 0x7f162523b830>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7f1624796de0>
fn_index=1
 inputs:
 |-<gradio.components.textbox.Textbox object at 0x7f1624796de0>
 outputs:
 |-<gradio.components.textbox.Textbox object at 0x7f1624796de0>
 |-<gradio.components.state.State object at 0x7f16247097c0>
fn_index=2
 inputs:
 |-<gradio.components.state.State object at 0x7f16247097c0>
 |-<gradio.components.chatbot.Chatbot object at 0x7f16247942f0>
 outputs:
 |-<gradio.components.chatbot.Chatbot object at 0x7f16247942f0>
fn_index=3
 inputs:
 |-<gradio.components.state.State object at 0x7f16247097c0>
 |-<gradio.components.state.State object at 0x7f1625234080>
 outputs:
 |-<gradio.components.state.State object at 0x7f16247644d0>
 |-<gradio.components.chatbot.Chatbot object at 0x7f16247942f0>
fn_index=4
 inputs:
 |-<gradio.components.